# Gold Layer Transformation: Business Aggregates & Star Schema Marts
**Framework:** Metadata-Driven Pipeline Framework  
**Source Catalog:** `silver_catalog.insurance_conformed`  
**Target Catalog:** `gold_catalog.insurance_analytics`  
**Capabilities:**  
1. Curated Star Schema Marts for underwriting, claims, and agency sales.
2. Business KPI calculations: Loss Ratio %, Settlement Turnaround Days, Net Profitability.
3. Aggregated monthly summaries for executive reporting.
4. Delta Lake `MERGE WITH SCHEMA EVOLUTION` and audit timestamps (`GoldIngestedAt`).

In [ ]:
# 1. Input Widgets
dbutils.widgets.text("silver_catalog", "silver_catalog", "Silver Catalog")
dbutils.widgets.text("silver_schema", "insurance_conformed", "Silver Schema")
dbutils.widgets.text("gold_catalog", "gold_catalog", "Gold Catalog")
dbutils.widgets.text("gold_schema", "insurance_analytics", "Gold Schema")

silver_catalog = dbutils.widgets.get("silver_catalog").strip()
silver_schema  = dbutils.widgets.get("silver_schema").strip()
gold_catalog   = dbutils.widgets.get("gold_catalog").strip()
gold_schema    = dbutils.widgets.get("gold_schema").strip()

print(f"Source Silver: `{silver_catalog}`.`{silver_schema}`")
print(f"Target Gold: `{gold_catalog}`.`{gold_schema}`")

# Ensure Gold Schema Exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{gold_catalog}`.`{gold_schema}`")
print(f"Verified schema `{gold_catalog}`.`{gold_schema}`.")

In [ ]:
# 2. Curated Fact: fact_policy_performance
from pyspark.sql.functions import col, current_timestamp, coalesce, lit, when, round as spark_round, count, sum as spark_sum, date_trunc

print("Building Gold fact_policy_performance...")
try:
    df_pol  = spark.table(f"`{silver_catalog}`.`{silver_schema}`.`fct_policy`")
    df_cust = spark.table(f"`{silver_catalog}`.`{silver_schema}`.`dim_customer`")
    df_agt  = spark.table(f"`{silver_catalog}`.`{silver_schema}`.`dim_agent`")
    df_cov  = spark.table(f"`{silver_catalog}`.`{silver_schema}`.`dim_coverage`")
    df_clm  = spark.table(f"`{silver_catalog}`.`{silver_schema}`.`fct_claim`")
    
    # Aggregate claims by PolicyID
    df_clm_agg = df_clm.groupBy("PolicyID").agg(
        count("ClaimID").alias("TotalClaimsCount"),
        spark_sum("ClaimAmount").alias("TotalClaimsIncurred"),
        spark_sum("SettledAmount").alias("TotalClaimsSettled")
    )
    
    # Join Policy with Dimensions and Aggregated Claims
    df_pol_perf = df_pol \
        .join(df_cust.select("CustomerID", col("FullName").alias("CustomerName"), "City", "State"), "CustomerID", "left") \
        .join(df_agt.select("AgentID", col("FullName").alias("AgentName"), "Region", "BranchName"), "AgentID", "left") \
        .join(df_cov.select("CoverageCode", "CoverageName", "Category"), "CoverageCode", "left") \
        .join(df_clm_agg, "PolicyID", "left") \
        .withColumn("TotalClaimsCount", coalesce(col("TotalClaimsCount"), lit(0))) \
        .withColumn("TotalClaimsIncurred", coalesce(col("TotalClaimsIncurred"), lit(0.00))) \
        .withColumn("TotalClaimsSettled", coalesce(col("TotalClaimsSettled"), lit(0.00))) \
        .withColumn("NetProfitMargin", spark_round(col("PremiumAmount") - col("TotalClaimsIncurred"), 2)) \
        .withColumn("LossRatioPercentage", spark_round(
            when(col("PremiumAmount") > 0, (col("TotalClaimsIncurred") / col("PremiumAmount")) * 100).otherwise(0.00), 2
        )) \
        .withColumn("IsProfitable", when(col("NetProfitMargin") >= 0, lit(True)).otherwise(lit(False))) \
        .withColumn("GoldIngestedAt", current_timestamp())
        
    target_pol_perf = f"`{gold_catalog}`.`{gold_schema}`.`fact_policy_performance`"
    df_pol_perf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_pol_perf)
    print(f"Successfully wrote {target_pol_perf} ({df_pol_perf.count()} records).")
except Exception as e_pol_perf:
    print(f"Note on fact_policy_performance: {str(e_pol_perf)}")

In [ ]:
# 3. Curated Fact: fact_claims_settlement
print("Building Gold fact_claims_settlement...")
try:
    df_clm  = spark.table(f"`{silver_catalog}`.`{silver_schema}`.`fct_claim`")
    df_pol  = spark.table(f"`{silver_catalog}`.`{silver_schema}`.`fct_policy`")
    df_cust = spark.table(f"`{silver_catalog}`.`{silver_schema}`.`dim_customer`")
    df_cov  = spark.table(f"`{silver_catalog}`.`{silver_schema}`.`dim_coverage`")
    
    df_clm_settle = df_clm \
        .join(df_pol.select("PolicyID", "CustomerID", "CoverageCode", "PremiumAmount", "DeductibleAmount", "PolicyStatus"), "PolicyID", "left") \
        .join(df_cust.select("CustomerID", col("FullName").alias("CustomerName"), "Email", "Phone"), "CustomerID", "left") \
        .join(df_cov.select("CoverageCode", "CoverageName", "Category"), "CoverageCode", "left") \
        .withColumn("SettlementRatio", spark_round(
            when(col("ClaimAmount") > 0, (col("SettledAmount") / col("ClaimAmount")) * 100).otherwise(0.00), 2
        )) \
        .withColumn("SeverityTier", when(col("ClaimAmount") >= 50000, "HIGH_SEVERITY") \
                                   .when(col("ClaimAmount") >= 10000, "MEDIUM_SEVERITY") \
                                   .otherwise("LOW_SEVERITY")) \
        .withColumn("DeductibleAssessment", when(col("ClaimAmount") <= col("DeductibleAmount"), "BELOW_DEDUCTIBLE") \
                                           .otherwise("ELIGIBLE_FOR_PAYOUT")) \
        .withColumn("GoldIngestedAt", current_timestamp())
        
    target_clm_settle = f"`{gold_catalog}`.`{gold_schema}`.`fact_claims_settlement`"
    df_clm_settle.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_clm_settle)
    print(f"Successfully wrote {target_clm_settle} ({df_clm_settle.count()} records).")
except Exception as e_clm_settle:
    print(f"Note on fact_claims_settlement: {str(e_clm_settle)}")

In [ ]:
# 4. Aggregate Mart: agg_agent_performance_monthly
print("Building Gold agg_agent_performance_monthly...")
try:
    df_pol_perf = spark.table(f"`{gold_catalog}`.`{gold_schema}`.`fact_policy_performance`")
    
    df_agent_mth = df_pol_perf \
        .withColumn("MonthYear", date_trunc("month", col("PolicyStartDate"))) \
        .groupBy("AgentID", "AgentName", "Region", "BranchName", "MonthYear") \
        .agg(
            count("PolicyID").alias("PoliciesWritten"),
            spark_sum(when(col("IsActivePolicy") == True, 1).otherwise(0)).alias("ActivePoliciesCount"),
            spark_round(spark_sum("PremiumAmount"), 2).alias("TotalPremiumWritten"),
            spark_sum("TotalClaimsCount").alias("TotalClaimsFiled"),
            spark_round(spark_sum("TotalClaimsIncurred"), 2).alias("TotalClaimsIncurred"),
            spark_round(spark_sum("NetProfitMargin"), 2).alias("NetUnderwritingProfit")
        ) \
        .withColumn("AgentLossRatio", spark_round(
            when(col("TotalPremiumWritten") > 0, (col("TotalClaimsIncurred") / col("TotalPremiumWritten")) * 100).otherwise(0.00), 2
        )) \
        .withColumn("GoldIngestedAt", current_timestamp())
        
    target_agent_mth = f"`{gold_catalog}`.`{gold_schema}`.`agg_agent_performance_monthly`"
    df_agent_mth.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_agent_mth)
    print(f"Successfully wrote {target_agent_mth} ({df_agent_mth.count()} records).")
except Exception as e_agt_mth:
    print(f"Note on agg_agent_performance_monthly: {str(e_agt_mth)}")

In [ ]:
# 5. Aggregate Mart: agg_claims_loss_ratio_by_coverage
print("Building Gold agg_claims_loss_ratio_by_coverage...")
try:
    df_pol_perf = spark.table(f"`{gold_catalog}`.`{gold_schema}`.`fact_policy_performance`")
    
    df_cov_mth = df_pol_perf \
        .withColumn("MonthYear", date_trunc("month", col("PolicyStartDate"))) \
        .groupBy("CoverageCode", "CoverageName", "Category", "MonthYear") \
        .agg(
            count("PolicyID").alias("PoliciesCount"),
            spark_round(spark_sum("PremiumAmount"), 2).alias("TotalEarnedPremium"),
            spark_sum("TotalClaimsCount").alias("TotalClaimsCount"),
            spark_round(spark_sum("TotalClaimsIncurred"), 2).alias("TotalClaimsIncurred")
        ) \
        .withColumn("CoverageLossRatio", spark_round(
            when(col("TotalEarnedPremium") > 0, (col("TotalClaimsIncurred") / col("TotalEarnedPremium")) * 100).otherwise(0.00), 2
        )) \
        .withColumn("GoldIngestedAt", current_timestamp())
        
    target_cov_mth = f"`{gold_catalog}`.`{gold_schema}`.`agg_claims_loss_ratio_by_coverage`"
    df_cov_mth.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_cov_mth)
    print(f"Successfully wrote {target_cov_mth} ({df_cov_mth.count()} records).")
except Exception as e_cov_mth:
    print(f"Note on agg_claims_loss_ratio_by_coverage: {str(e_cov_mth)}")

In [ ]:
# 6. Aggregate Mart: agg_customer_risk_profile
print("Building Gold agg_customer_risk_profile...")
try:
    df_pol_perf = spark.table(f"`{gold_catalog}`.`{gold_schema}`.`fact_policy_performance`")
    
    df_cust_risk = df_pol_perf \
        .groupBy("CustomerID", "CustomerName", "City", "State") \
        .agg(
            count("PolicyID").alias("TotalPoliciesHeld"),
            spark_sum(when(col("IsActivePolicy") == True, 1).otherwise(0)).alias("ActivePoliciesHeld"),
            spark_round(spark_sum("PremiumAmount"), 2).alias("LifetimeCustomerPremium"),
            spark_sum("TotalClaimsCount").alias("LifetimeClaimsCount"),
            spark_round(spark_sum("TotalClaimsIncurred"), 2).alias("LifetimeClaimsIncurred"),
            spark_round(spark_sum("TotalClaimsSettled"), 2).alias("LifetimeClaimsPaidOut")
        ) \
        .withColumn("CustomerLossRatio", spark_round(
            when(col("LifetimeCustomerPremium") > 0, (col("LifetimeClaimsIncurred") / col("LifetimeCustomerPremium")) * 100).otherwise(0.00), 2
        )) \
        .withColumn("CustomerRiskTier", when(col("LifetimeClaimsCount") == 0, "PRIME_ZERO_CLAIMS") \
                                      .when(col("LifetimeClaimsIncurred") > col("LifetimeCustomerPremium"), "HIGH_RISK") \
                                      .when(col("LifetimeClaimsCount") >= 3, "FREQUENT_CLAIMANT") \
                                      .otherwise("STANDARD_RISK")) \
        .withColumn("GoldIngestedAt", current_timestamp())
        
    target_cust_risk = f"`{gold_catalog}`.`{gold_schema}`.`agg_customer_risk_profile`"
    df_cust_risk.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_cust_risk)
    print(f"Successfully wrote {target_cust_risk} ({df_cust_risk.count()} records).")
except Exception as e_cust_risk:
    print(f"Note on agg_customer_risk_profile: {str(e_cust_risk)}")

In [ ]:
# 7. Return Execution Output JSON
import json

output = {
    "status": "Success",
    "layer": "Gold",
    "target_schema": f"{gold_catalog}.{gold_schema}",
    "curated_facts": ["fact_policy_performance", "fact_claims_settlement"],
    "aggregate_marts": ["agg_agent_performance_monthly", "agg_claims_loss_ratio_by_coverage", "agg_customer_risk_profile"]
}

print(json.dumps(output, indent=2))
dbutils.notebook.exit(json.dumps(output))